In [3]:
import pandas as pd
import os

# Load the dataset from the provided CSV file
retail_data_file_path = 'OnlineRetail.csv'
df_retail = pd.read_csv(retail_data_file_path, encoding = 'ISO-8859-1')

print("\n===============================")
print("# Online Retail Customer Segmentation Dataset")
print("===============================")
print(df_retail.head())
print(df_retail.info())
print(df_retail.isnull().sum())


# Online Retail Customer Segmentation Dataset
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------  

## Data Cleaning and Preprocessing

Based on the `df_retail.info()` and `df_retail.isnull().sum()` outputs, we need to perform the following cleaning steps:

1.  **Handle Missing Values**:
    *   `Description`: Fill missing descriptions (or drop rows). Since the number of missing descriptions is small (1454 out of 541909), and they might be associated with other invalid entries, we will drop them.
    *   `CustomerID`: Drop rows where `CustomerID` is null. This is crucial for customer segmentation, as we need a valid customer ID for each transaction.
2.  **Convert Data Types**: Convert `InvoiceDate` to a datetime object.
3.  **Filter Out Invalid Entries**: Remove rows where `Quantity` is less than or equal to 0, as negative quantities usually indicate returns and are not relevant for typical purchase analysis in customer segmentation. Similarly, remove `UnitPrice` less than 0.
4.  **Calculate Total Price**: Create a new column `TotalPrice` by multiplying `Quantity` and `UnitPrice`.

In [4]:
# Make a copy to work on, preserving the original dataframe
df = df_retail.copy()

# 1. Handle Missing Values
# Drop rows with missing Description
df.dropna(subset=['Description'], inplace=True)

# Drop rows with missing CustomerID
df.dropna(subset=['CustomerID'], inplace=True)

# Convert CustomerID to integer type after dropping NaNs
df['CustomerID'] = df['CustomerID'].astype(int)

print("DataFrame after handling missing values:")
print(df.isnull().sum())
print(f"New DataFrame shape: {df.shape}")

DataFrame after handling missing values:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64
New DataFrame shape: (406829, 8)


In [5]:
# 2. Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("DataFrame info after InvoiceDate conversion:")
print(df.info())

DataFrame info after InvoiceDate conversion:
<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    406829 non-null  object        
 1   StockCode    406829 non-null  object        
 2   Description  406829 non-null  object        
 3   Quantity     406829 non-null  int64         
 4   InvoiceDate  406829 non-null  datetime64[ns]
 5   UnitPrice    406829 non-null  float64       
 6   CustomerID   406829 non-null  int64         
 7   Country      406829 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 27.9+ MB
None


In [6]:
# 3. Filter out invalid entries (Quantity <= 0 or UnitPrice < 0)
# Transactions with Quantity <= 0 are returns or adjustments
df = df[df['Quantity'] > 0]

# Transactions with UnitPrice < 0 are also typically returns or errors
df = df[df['UnitPrice'] >= 0]

print("DataFrame after filtering out invalid quantities and prices:")
print(f"New DataFrame shape: {df.shape}")

DataFrame after filtering out invalid quantities and prices:
New DataFrame shape: (397924, 8)


In [7]:
# 4. Calculate TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print("DataFrame head with TotalPrice:")
display(df.head())

print("Final DataFrame info after all cleaning steps:")
print(df.info())

DataFrame head with TotalPrice:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


Final DataFrame info after all cleaning steps:
<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    397924 non-null  object        
 1   StockCode    397924 non-null  object        
 2   Description  397924 non-null  object        
 3   Quantity     397924 non-null  int64         
 4   InvoiceDate  397924 non-null  datetime64[ns]
 5   UnitPrice    397924 non-null  float64       
 6   CustomerID   397924 non-null  int64         
 7   Country      397924 non-null  object        
 8   TotalPrice   397924 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 30.4+ MB
None


In [13]:
# Calculate Recency, Frequency, Monetary (RFM) values

# 1. Determine the snapshot date (one day after the last transaction date)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Group by CustomerID to calculate RFM metrics
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda date: (snapshot_date - date.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

print("RFM DataFrame head:")
display(rfm.head())

print("RFM DataFrame info:")
print(rfm.info())

RFM DataFrame head:


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


RFM DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4339 entries, 0 to 4338
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CustomerID  4339 non-null   int64  
 1   Recency     4339 non-null   int64  
 2   Frequency   4339 non-null   int64  
 3   Monetary    4339 non-null   float64
dtypes: float64(1), int64(3)
memory usage: 135.7 KB
None


In [12]:
# Calculate Recency, Frequency, Monetary (RFM) values

# 1. Determine the snapshot date (one day after the last transaction date)
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Group by CustomerID to calculate RFM metrics
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda date: (snapshot_date - date.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

print("RFM DataFrame head:")
display(rfm.head())

print("RFM DataFrame info:")
print(rfm.info())

RFM DataFrame head:


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


RFM DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4339 entries, 0 to 4338
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CustomerID  4339 non-null   int64  
 1   Recency     4339 non-null   int64  
 2   Frequency   4339 non-null   int64  
 3   Monetary    4339 non-null   float64
dtypes: float64(1), int64(3)
memory usage: 135.7 KB
None


## CRISP-DM for Online Retail Customer Classification

To apply a classification model, we first need to define a target variable. For the 'Online Retail' dataset, we can create a binary target, for example, classifying customers as 'High-Value' or 'Standard-Value' based on their monetary contributions.

We'll use the `rfm` DataFrame, which contains Recency, Frequency, and Monetary values for each customer. From this, we'll derive our target variable.

### 1. Business Understanding & Data Understanding (Initial Steps)

These steps were covered in the previous data loading, cleaning, and RFM analysis. We've cleaned the data and calculated RFM metrics, which will serve as our features.

### 2. Data Preparation: Define Target Variable

We'll define 'High-Value Customer' as customers whose `Monetary` value falls into the top quartile (or top 25%) of all customers. This creates a binary classification problem.

In [14]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create a copy of the RFM data for classification
rfm_classification = rfm.copy()

# Define 'High-Value Customer' as top 25% of Monetary value
monetary_threshold = rfm_classification['Monetary'].quantile(0.75)
rfm_classification['HighValueCustomer'] = (rfm_classification['Monetary'] >= monetary_threshold).astype(int)

print("RFM data with 'HighValueCustomer' target variable:")
display(rfm_classification.head())
print(rfm_classification['HighValueCustomer'].value_counts())

# Define features (X) and target (y)
X = rfm_classification[['Recency', 'Frequency', 'Monetary']]
y = rfm_classification['HighValueCustomer']

print("\nFeatures (X) head:")
display(X.head())
print("\nTarget (y) head:")
display(y.head())

RFM data with 'HighValueCustomer' target variable:


,CustomerID,Recency,Frequency,Monetary,HighValueCustomer
0,12346,326,1,77183.60,1
1,12347,2,7,4310.00,1
2,12348,75,4,1797.24,1
3,12349,19,1,1757.55,1
4,12350,310,1,334.40,0


HighValueCustomer
0    3254
1    1085
Name: count, dtype: int64

Features (X) head:


,Recency,Frequency,Monetary
0,326,1,77183.60
1,2,7,4310.00
2,75,4,1797.24
3,19,1,1757.55
4,310,1,334.40



Target (y) head:


,HighValueCustomer
0,1
1,1
2,1
3,1
4,0


### 3. Data Preparation: Feature Scaling and Train-Test Split

We'll scale our numerical features (Recency, Frequency, Monetary) to ensure they contribute equally to the model, and then split the data into training and testing sets.

In [15]:
# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("Scaled Features (X) head:")
display(X_scaled_df.head())

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y # stratify to maintain class balance
)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Scaled Features (X) head:


,Recency,Frequency,Monetary
0,2.334858,-0.424675,8.359634
1,-0.905199,0.354080,0.251046
2,-0.175186,-0.035297,-0.028546
3,-0.735196,-0.424675,-0.032963
4,2.174855,-0.424675,-0.191315



Shape of X_train: (3471, 3)
Shape of X_test: (868, 3)
Shape of y_train: (3471,)
Shape of y_test: (868,)


### 4. Modeling: Model Training

We'll use a RandomForestClassifier, a popular and robust algorithm for classification tasks.

In [16]:
# Model Training
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print("RandomForestClassifier model trained successfully.")

RandomForestClassifier model trained successfully.


### 5. Evaluation: Make Predictions and Evaluate

We'll evaluate the model's performance on the test set using accuracy and a classification report.

In [18]:
# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature Importance
feature_importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importances:")
display(feature_importances)


Model Accuracy: 0.9988

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       651
           1       1.00      1.00      1.00       217

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868


Feature Importances:


,0
Monetary,0.694153
Frequency,0.249817
Recency,0.056030
